In [35]:
import joblib
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config
from tempfile import TemporaryDirectory
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import StackingClassifier
import xgboost as xgb
from xgboost import XGBClassifier
#from lightgbm import LGBMClassifier
#from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn import metrics
import seaborn as sns
import numpy as np
import warnings

In [36]:
target_column = "health_condition"

**Hard Voting**

In [37]:
model_names = ["catboost", "lgbm", "random_forest", "xgboost"]

baseline_dfs = []

for model_name in model_names:
    df = pd.read_csv(f"results/{model_name}_baseline.csv")
    df["model"] = model_name
    baseline_dfs.append(df)

baseline_dfs[0]

df = pd.concat(baseline_dfs, axis=0)
df

,id,health_condition,model
0,690088,unhealthy,catboost
1,690089,unhealthy,catboost
2,690090,at-risk,catboost
3,690091,at-risk,catboost
4,690092,unhealthy,catboost
...,...,...,...
295748,985836,fit,xgboost
295749,985837,at-risk,xgboost
295750,985838,unhealthy,xgboost
295751,985839,at-risk,xgboost


In [38]:
result = df.groupby('id')['health_condition'].agg(pd.Series.mode)
result

id
690088    unhealthy
690089    unhealthy
690090      at-risk
690091      at-risk
690092    unhealthy
            ...    
985836          fit
985837      at-risk
985838    unhealthy
985839      at-risk
985840    unhealthy
Name: health_condition, Length: 295753, dtype: object

In [39]:
df_submission = baseline_dfs[0].copy()
df_submission.pop('model')
df_submission = (
    baseline_dfs[0]
    .drop(columns='model')
    .drop(columns='health_condition', errors='ignore')
    .merge(result.rename('health_condition'), on='id', how='left')
)

In [40]:
df_submission.to_csv(f'results/hard_vote_{'_'.join(model_names)}.csv', index=False)
df_submission

,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy
...,...,...
295748,985836,fit
295749,985837,at-risk
295750,985838,unhealthy
295751,985839,at-risk
